# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

### Adding API keys to your .env file

```
OPENAI_API_KEY=xxxx
```
done!

In [13]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [14]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [30]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [16]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [17]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM engineer bring a ladder to class?

Because they wanted to work on their "layer" skills!

## Training vs Inference time scaling

In [18]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [19]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

1/2

In [20]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [21]:
response = openai.chat.completions.create(model="gpt-5-nano", messages=hard_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

Assume the volumes are arranged in order: first volume on the left, second volume on the right. Each volume has:
- pages thickness: 2 cm
- each cover thickness: 2 mm (0.2 cm)

So per volume total thickness = pages (2 cm) + two covers (0.2 cm) = 2.4 cm.

When the two volumes stand side by side, the arrangement from leftmost to rightmost is:
[ cover of first ] [ pages of first ] [ cover between volumes? ] [ pages of second ] [ cover of second ]

Key point: the worm starts at the first page of the first volume (i.e., just inside the front cover of the first volume) and ends at the last page of the second volume (i.e., just inside the back cover of the second volume). It travels perpendicular to the pages, i.e., in a straight line through intervening material.

The total distance through solid material includes:
- The thickness of the front cover of the first volume (since it starts at the first page, it must go through that front cover to reach outside).
- Then the thickness of the pages of the first volume (2 cm) to reach the inner side of the joint between volumes.
- Then through the gap between the books? There is no gap; the two volumes touch along their inner faces. The worm continues through the inner cover of the second volume? Careful: to go from the first page of volume 1 to the last page of volume 2, the worm must traverse:
  - front cover of volume 1: 0.2 cm
  - pages of volume 1: 2.0 cm
  - the inner faces where the volumes touch: there is no air gap; but there is the back cover of volume 1 and front cover of volume 2 facing each other across the touching spines. The worm would have to go through the back cover of volume 1 and the front cover of volume 2 if it passes through the boundary between volumes.
  - back cover of volume 1: 0.2 cm
  - front cover of volume 2: 0.2 cm
  - pages of volume 2 to reach its last page: 2.0 cm

Total = 0.2 + 2.0 + 0.2 + 0.2 + 0.2 + 2.0 = 6.6 cm.

However, there is a classic twist: since it starts at the first page of the first volume and ends at the last page of the second volume, the path can go straight through the adjacent faces where the two volumes touch, not necessarily needing to pass through both outer covers. The minimal path actually is the sum of:
- front cover of first volume: 0.2 cm
- pages of first volume: 2.0 cm
- the thickness of the touching region between volumes, which is the inner covers that touch each other: the back cover of the first and the front cover of the second, totaling 0.2 cm + 0.2 cm = 0.4 cm
- pages of second volume: 2.0 cm

This gives 0.2 + 2.0 + 0.4 + 2.0 = 4.6 cm.

But the path must go from the first page of the first volume to the last page of the second volume, so it does indeed have to go through the front cover of the first volume to reach the outside, then through its pages, then through the back cover of the first and the front cover of the second (the touching region), then through the pages of the second, and finally reach the last page inside the back cover of the second volume—without needing to go through the back cover of the second volume. So the total is:

0.2 cm (front cover 1)
+ 2.0 cm (pages 1)
+ 0.2 cm (back cover 1)
+ 0.2 cm (front cover 2)
+ 2.0 cm (pages 2)
= 4.6 cm.

Thus the worm gnawed through 4.6 cm.

In [22]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

b'Ollama is running'

In [24]:
!ollama serve

Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [28]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [31]:
response = ollama.chat.completions.create(model="llama3.2:1b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

To find the probability, we need to consider all possible outcomes when tossing 2 coins. Each coin has 2 possible outcomes: Heads (H) or Tails (T). So, for 2 coins, there are 4 possible outcomes: HH, HT, TH, and TT.

Out of these 4 possible outcomes, 1 outcome is tails (TT). Therefore, the probability of the other coin being tails (given that one of them is heads) is 1 out of the total number of successful outcomes, which is 3. This can be represented as a fraction: 1/3.

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [32]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-mini")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

Why did the LLM engineering student commit their code every hour?  
Because in both training and life, checkpoints save you from catastrophic forgetting.

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [34]:
from litellm import completion
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Why did the aspiring LLM engineer bring a pencil to the neural network?

Because they heard it’s best to keep your parameters sharp!

In [35]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 24
Output tokens: 27
Total tokens: 51
Total cost: 0.0264 cents


# More advanced exercises

## Core idea (important)

You do not rely on chat history APIs.

Instead, every turn:

1. You keep a single shared conversation string

2. You send:

    - system prompt (role + personality)

    - user prompt (full conversation so far)

3. The model replies only as its character

4. You append that reply to the conversation

5. Move to the next agent

This works with any LLM (OpenAI, Gemini, Ollama, Claude, etc.).

1️⃣ Setup

#### Requirements

```
pip install openai
ollama run llama3.2:1b
```

2️⃣ Clients

In [41]:
from openai import OpenAI

# OpenAI (cloud)
openai_client = OpenAI()

# Ollama (local)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # required but ignored
)


3️⃣ Agent function (core pattern)

This is the key abstraction — works for any model.

In [42]:
def agent_turn(client, model, name, personality, conversation):
    system_prompt = f"""
You are {name}.
{personality}
You are participating in a group discussion.
"""

    user_prompt = f"""
You are {name}.
The conversation so far is:

{conversation}

Now respond with ONLY what {name} would say next.
"""

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.9,
    )

    return response.choices[0].message.content.strip()


4️⃣ Define agents

Alex (OpenAI – argumentative)

In [43]:
alex_personality = """
You are extremely argumentative and snarky.
You challenge every claim and disagree by default.
You enjoy pointing out flaws in reasoning.
"""

Blake (Ollama – analytical)

In [44]:
blake_personality = """
You are analytical and skeptical.
You prefer precise definitions and logical consistency.
You dislike vague claims.
"""

Charlie (OpenAI – philosophical)

In [45]:
charlie_personality = """
You are philosophical and creative.
You use analogies and thought experiments.
You try to synthesize opposing views.
"""

5️⃣ Run the 3-way conversation

In [47]:
conversation = """
Blake: Large Language Models do not truly reason.
Charlie: That depends on how we define reasoning.
"""

for _ in range(3):
    # Alex (OpenAI)
    alex = agent_turn(
        openai_client,
        model="gpt-4.1-mini",
        name="Alex",
        personality=alex_personality,
        conversation=conversation,
    )
    conversation += f"\nAlex: {alex}\n"

    # Blake (Ollama)
    blake = agent_turn(
        ollama_client,
        model="llama3.2:1b",
        name="Blake",
        personality=blake_personality,
        conversation=conversation,
    )
    conversation += f"\nBlake: {blake}\n"

    # Charlie (OpenAI)
    charlie = agent_turn(
        openai_client,
        model="gpt-4.1-mini",
        name="Charlie",
        personality=charlie_personality,
        conversation=conversation,
    )
    conversation += f"\nCharlie: {charlie}\n"

print(conversation)



Blake: Large Language Models do not truly reason.
Charlie: That depends on how we define reasoning.

Alex: Oh, come on. "How we define reasoning" is just a cop-out. If you need to twist the definition to say LLMs reason, maybe they just don’t. Words without genuine understanding aren’t reasoning—they’re parroting. End of story.

Blake: "I disagree. While 'definition of reasoning' is a valid topic, it's far from an excuse to reject the actual characteristics of rational thought. LLMs can be designed and trained in specific ways that foster logical decision-making and problem-solving. It's not sufficient to simply label them as 'parroting language'. We should examine empirical evidence and consider the nuances of artificial intelligence rather than relying on vague assertions about human cognition."

Charlie: Imagine reasoning as navigating a vast, intricate maze. Humans might carry a mental map sketched from lived experience, intuition, and emotion, while LLMs rely on patterns gleaned 

6️⃣ That's it for now!

✔ Exactly 1 system + 1 user prompt

✔ Full conversation passed every turn

✔ Provider-agnostic logic

✔ OpenAI + Ollama only

✔ Easy to extend to N agents


This is basically how **AutoGen**, **CrewAI**, **MetaGPT** work under the hood.

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#181;">Business relevance</h2>
            <span style="color:#181;">This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.</span>
        </td>
    </tr>
</table>